# HRM-Text-1B generation with KerasHub

This inference-only notebook shows how to run the official Apache-2.0 `sapientinc/HRM-Text-1B` checkpoint with KerasHub: load the model, construct PrefixLM prompts, configure decoding, and generate complete answers. A Colab T4 is sufficient; expect paragraph-length responses to take a few minutes because every generated token traverses 128 logical recurrent cache slots.

> **Checkpoint setup.** HRM-Text-1B is not yet distributed as a hosted KerasHub preset. To keep the notebook reproducible, the setup checks out an immutable KerasHub source revision and converts the official checkpoint in memory. The rest of the notebook uses the normal KerasHub model and sampler APIs.

## Tokenizer vocabulary and inference contract

HRM-Text inherits a Qwen-style vocabulary, but these examples actively use only the eight tokens below. The notebook explicitly formats every prompt as `<|im_start|><condition>instruction<|im_end|>`; the model then generates a response and may end it with `<|box_end|>`.

| Token | Active inference role |
| --- | --- |
| `<|im_start|>` | Explicit start of the PrefixLM prompt envelope. |
| `<|im_end|>` | Explicit end of the PrefixLM prompt/instruction block. |
| `<|box_end|>` | Checkpoint response terminator and generation stop token. |
| `<|endoftext|>` | KerasHub padding token only; it is not prompt or response content. |
| `<|object_ref_start|>` | `direct` condition. |
| `<|object_ref_end|>` | `cot` condition. |
| `<|quad_start|>` | `noisy` condition. |
| `<|quad_end|>` | `synth` condition. |

The rest of the checkpoint vocabulary is inherited but unsupported in these examples: legacy `<|PAD|>` and `<|direct|>`, `<|cot|>`, `<|noisy|>`, `<|synth|>` labels; reserved `<|box_start|>`; vision `<|vision_start|>`, `<|vision_end|>`, `<|vision_pad|>`, `<|image_pad|>`, `<|video_pad|>`; fill-in-the-middle `<|fim_prefix|>`, `<|fim_middle|>`, `<|fim_suffix|>`, `<|fim_pad|>`; repository `<|repo_name|>`, `<|file_sep|>`; tool `<tool_call>`, `</tool_call>`, `<tool_response>`, `</tool_response>`; and reasoning `<think>`, `</think>`. `<tool_response>` is vocabulary only here: no tool-use protocol is enabled, and `<tool_result>` is not a checkpoint special token.

Responsibility is deliberately separated: this notebook selects condition tokens and formats the complete envelope; `HrmTextTokenizer` atomically encodes registered special-token strings; the HRM model consumes token IDs and `token_type_ids` (the PrefixLM mask). Neither the tokenizer nor model injects prompt tokens. The Hugging Face tokenizer remains only the checkpoint-conversion and parity/reference implementation.

## What makes this model different?

HRM-Text runs shared high-level (H) and low-level (L) Transformer stacks repeatedly. The official 1B configuration has 16 layers per stack, two H cycles, and three L cycles. During generation, each logical stack invocation needs its own KV-cache slot: `16 × 2 × (3 + 1) = 128`. The public API remains familiar: provide token IDs and masks, then call `generate()`.

In [ ]:
import gc
import json
import os
import subprocess
import time

# An immutable source revision makes this notebook reproducible. Update this
# value whenever a new implementation commit is pushed.
REPOSITORY = "https://github.com/pzarzycki/keras-hub.git"
KERAS_HUB_REVISION = "3fd46ba9e4360fbfe808d99f3c97d15aba4ae285"
MODEL_ID = "sapientinc/HRM-Text-1B"
MODEL_REVISION = "9f082d68b8cd0ebc56e33f1c88c45609174c272c"
WORKDIR = "/content/keras-hub"

subprocess.run(["bash", "-lc", "command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"
if os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "-C", WORKDIR, "fetch", "origin"], check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY, WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "checkout", "--detach", KERAS_HUB_REVISION], check=True)
# Pin Transformers without globally upgrading Colab's compiled scientific stack.
subprocess.run(["uv", "pip", "install", "--system", "-q", "transformers==5.9.0", "safetensors"], check=True)
subprocess.run(["uv", "pip", "install", "--system", "-q", "-e", WORKDIR], check=True)

# Select the Keras backend before importing Keras. The conversion below uses
# PyTorch because it reads the original checkpoint directly on the GPU.
os.environ["KERAS_BACKEND"] = "torch"


In [ ]:
import os
os.chdir(WORKDIR)

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")

torch.set_default_device("cuda")
import keras
import keras_hub
from tools.checkpoint_conversion.convert_hrm_text_checkpoints import (
    convert_tokenizer,
    convert_weights,
    create_backbone,
)

# Make the notebook safe to rerun in a runtime that has held another
# HRM-Text model. Conversion briefly needs memory for both model copies.
for name in ("model", "backbone", "hf_model", "restored"):
    globals().pop(name, None)
gc.collect()
keras.backend.clear_session()
torch.cuda.empty_cache()

keras.config.set_dtype_policy("float32")
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.float32
).cuda().eval()

backbone = create_backbone(hf_model.config)
convert_weights(backbone, hf_model)
tokenizer = convert_tokenizer(hf_tokenizer)
model = keras_hub.models.HrmTextCausalLM(backbone, preprocessor=None)
# Generation only needs the converted Keras model. Release the source model
# so longer responses have more GPU memory available.
del hf_model
torch.cuda.empty_cache()
print(f"Keras backend: {keras.config.backend()}")
print(f"Logical KV-cache slots: {backbone.cache_slots}")


## Build a PrefixLM prompt

The released model is a base pre-alignment model, not an instruction-tuned chat model. The notebook, rather than KerasHub, explicitly constructs `<|im_start|><condition>instruction<|im_end|>`. Upstream recommends `synth,cot` for reasoning and open-ended generation, and `direct` with 2–8 few-shot examples for short-form QA or extraction.

In [ ]:
ACTIVE_INFERENCE_TOKENS = {
    "start": "<|im_start|>",
    "prefix_end": "<|im_end|>",
    "end": "<|box_end|>",
    "pad": "<|endoftext|>",
    "direct": "<|object_ref_start|>",
    "cot": "<|object_ref_end|>",
    "noisy": "<|quad_start|>",
    "synth": "<|quad_end|>",
}
for role, special_token in ACTIVE_INFERENCE_TOKENS.items():
    keras_ids = np.asarray(tokenizer([special_token])[0], dtype="int32").tolist()
    hf_ids = hf_tokenizer(special_token, add_special_tokens=False)["input_ids"]
    assert keras_ids == hf_ids == [tokenizer.token_to_id(special_token)], role

CONDITIONS = {
    "direct": "<|object_ref_start|>",
    "cot": "<|object_ref_end|>",
    "noisy": "<|quad_start|>",
    "synth": "<|quad_end|>",
}

def make_prompt(text, conditions=("synth", "cot")):
    """Return an upstream-format PrefixLM prompt."""
    condition_tokens = "".join(CONDITIONS[name] for name in conditions)
    return f"<|im_start|>{condition_tokens}{text}<|im_end|>"

for condition_name in CONDITIONS:
    prompt = make_prompt("Tokenization parity check.", (condition_name,))
    keras_ids = np.asarray(tokenizer([prompt])[0], dtype="int32").tolist()
    hf_ids = hf_tokenizer(prompt, add_special_tokens=False)["input_ids"]
    assert keras_ids == hf_ids, condition_name

print(make_prompt("Explain why the sky is blue."))


## Choose generation controls

Greedy decoding is deterministic and matches the official checkpoint example. Set `USE_SAMPLING = True` to explore seeded top-k sampling; `TEMPERATURE` controls randomness and `TOP_K` limits the candidate set. `MAX_NEW_TOKENS = 64` is long enough for a paragraph while remaining practical on a T4.

In [ ]:
USE_SAMPLING = False
MAX_NEW_TOKENS = 64
TOP_K = 40
TEMPERATURE = 0.8
SEED = 42

if USE_SAMPLING:
    sampler = keras_hub.samplers.TopKSampler(
        k=TOP_K, temperature=TEMPERATURE, seed=SEED
    )
    sampler_name = f"top-k (k={TOP_K}, temperature={TEMPERATURE}, seed={SEED})"
else:
    sampler = "greedy"
    sampler_name = "greedy"

model.compile(sampler=sampler)
print(f"Sampler: {sampler_name}; default maximum: {MAX_NEW_TOKENS} new tokens")


In [ ]:
from IPython.display import Markdown, display

def generate(text, conditions=("synth", "cot"), max_new_tokens=MAX_NEW_TOKENS):
    """Generate and return only the model's continuation plus metadata."""
    prompt = make_prompt(text, conditions)
    prompt_ids = np.asarray(tokenizer([prompt])[0], dtype="int32")
    reference_prompt_ids = hf_tokenizer(prompt, add_special_tokens=False)["input_ids"]
    assert prompt_ids.tolist() == reference_prompt_ids
    max_length = len(prompt_ids) + max_new_tokens
    token_ids = np.full((1, max_length), tokenizer.pad_token_id, dtype="int32")
    token_ids[0, : len(prompt_ids)] = prompt_ids
    padding_mask = np.zeros((1, max_length), dtype="int32")
    padding_mask[0, : len(prompt_ids)] = 1

    # All present tokens form one bidirectional PrefixLM context block.
    torch.cuda.synchronize()
    started = time.perf_counter()
    generated = model.generate(
        {
            "token_ids": token_ids,
            "padding_mask": padding_mask,
            "token_type_ids": padding_mask.copy(),
        },
        max_length=max_length,
        stop_token_ids=[tokenizer.end_token_id],
    )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    output_ids = keras.ops.convert_to_numpy(generated["token_ids"])[0]
    output_mask = keras.ops.convert_to_numpy(generated["padding_mask"])[0]
    valid_ids = output_ids[output_mask.astype(bool)]
    continuation_ids = valid_ids[len(prompt_ids):]
    stopped_on_eos = bool(
        len(continuation_ids)
        and continuation_ids[-1] == tokenizer.end_token_id
    )
    decode_ids = continuation_ids[:-1] if stopped_on_eos else continuation_ids
    answer = str(tokenizer.detokenize(decode_ids)).strip()
    return {
        "query": text,
        "conditions": list(conditions),
        "answer": answer,
        "generated_tokens": int(len(continuation_ids)),
        "stopped_on_eos": stopped_on_eos,
        "seconds": elapsed,
    }

def show_result(result):
    display(Markdown(f"**Query**\n\n{result['query']}\n\n**Model answer**\n\n{result['answer']}"))
    print(
        f"[{result['generated_tokens']} generated tokens; "
        f"EOS={result['stopped_on_eos']}; {result['seconds']:.1f} seconds]"
    )


In [ ]:
results = []
explanation = generate(
    "Explain why the sky is blue in a short paragraph.",
    conditions=("synth", "cot"),
)
results.append(explanation)
show_result(explanation)


In [ ]:
few_shot_query = (
    "Question: What is the largest ocean on Earth?\n"
    "Answer: Pacific Ocean.\n"
    "Question: What gas do plants absorb from the atmosphere?\n"
    "Answer: Carbon dioxide.\n"
    "Question: What is the capital of Japan?\nAnswer:"
)
short_qa = generate(few_shot_query, conditions=("direct",), max_new_tokens=32)
results.append(short_qa)
show_result(short_qa)


In [ ]:
health = generate(
    "Write two concise sentences explaining why regular exercise benefits human health.",
    conditions=("synth", "cot"),
)
results.append(health)
show_result(health)

with open("/content/hrm_text_1b_usage_results.json", "w") as f:
    json.dump(
        {
            "implementation_revision": KERAS_HUB_REVISION,
            "model_revision": MODEL_REVISION,
            "gpu": torch.cuda.get_device_name(0),
            "sampler": sampler_name,
            "cache_slots": backbone.cache_slots,
            "examples": results,
        },
        f,
        indent=2,
    )


## Save the converted model for reuse

Checkpoint conversion only needs to happen once. Save the converted model as a local Keras preset when you want to reuse it in another process or notebook. HRM-Text-1B is a base pre-alignment model rather than a chat model, so evaluate it for your task before relying on its outputs. When a maintained hosted preset becomes available, prefer it over a personal checkpoint copy.

In [ ]:
# model.save_to_preset("/content/hrm_text_1b")
# Later, load the local directory with:
# reloaded = HrmTextCausalLM.from_preset("/content/hrm_text_1b")
